In [1]:
import jax
import jax.numpy as jnp

def f(x):
    y = x + 1.0
    z = jnp.sin(y)
    return z **2

# jit包装
f_jit = jax.jit(f)

x = jnp.array(1.0)

# 1. 获取 jaxpr(JAX中间表示，计算图的文本形式)
jaxpr = jax.make_jaxpr(f)(x)
print(jaxpr)


{ lambda ; a:f32[]. let
    b:f32[] = add a 1.0:f32[]
    c:f32[] = sin b
    d:f32[] = integer_pow[y=2] c
  in (d,) }


In [2]:
# 获取HLO文本
hlo = f_jit.lower(x).compiler_ir()
print(hlo)


module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %0 = stablehlo.add %arg0, %cst : tensor<f32>
    %1 = stablehlo.sine %0 : tensor<f32>
    %2 = stablehlo.multiply %1, %1 : tensor<f32>
    return %2 : tensor<f32>
  }
}

